# Imports

In [3]:
import albumentations as A
from torch.utils.data import DataLoader
from segmentation_models_pytorch.losses import DiceLoss, SoftBCEWithLogitsLoss, JaccardLoss, FocalLoss

# Custom Library
import ActivationPrototypes_SARSeg.thesis_utils as utils
from ActivationPrototypes_SARSeg.thesis_utils import *

initialized = False

c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-06 11:30:26.788 | INFO     | ActivationPrototypes_SARSeg.config:<module>:11 - PROJ_ROOT path is: F:\Thesis\ActivationPrototypes_SARSeg


# Setup

In [31]:
from importlib import reload
reload(utils)
from ActivationPrototypes_SARSeg.thesis_utils import *

In [14]:
lmdb_path = "F:\\Thesis\\Datasets\\Big Earth\\Encoded-BigEarthNet"
parquet_path = "F:\\Thesis\Datasets\\Big Earth\\metadata.parquet"

In [10]:
if not initialized:
    train_matches, val_matches, test_matches = match_keys(parquet_path)
    initialized = True

In [11]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
],is_check_shapes=False)

print(f"Train size: {len(train_matches)}, Val size: {len(val_matches)}, Test size: {len(test_matches)}")

# Create datasets	
train_dataset = SARSegmentationDataset120(lmdb_path, train_matches[:], transform=transform)
val_dataset = SARSegmentationDataset120(lmdb_path, val_matches[:], transform=None)
test_dataset = SARSegmentationDataset120(lmdb_path, test_matches[:], transform=None)

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#Define the model metrcis and load model. 
num_train_img = len(train_dataset)
num_val_img = len(val_dataset)
num_test_img = len(test_dataset)

steps_per_epoch = num_train_img//batch_size
val_steps_per_epoch = num_val_img//batch_size
print("Steps per epoch: ", steps_per_epoch)
print("Validation steps per epoch: ", val_steps_per_epoch)

Train size: 237871, Val size: 122342, Test size: 119825
Opening LMDB environment ...
Opening LMDB environment ...
Opening LMDB environment ...
Steps per epoch:  14866
Validation steps per epoch:  7646


In [12]:
random_indices_train = random.sample(range(len(train_matches)), 10000)
random_indices_val = random.sample(range(len(val_matches)), 4000)
random_indices_test = random.sample(range(len(test_matches)), 4000)

random_train_matches = [train_matches[i] for i in random_indices_train]
random_val_matches = [val_matches[i] for i in random_indices_val]
random_test_matches = [test_matches[i] for i in random_indices_test]

train_dataset_short = SARSegmentationDataset120(lmdb_path, random_train_matches[:], transform=transform)
val_dataset_short = SARSegmentationDataset120(lmdb_path, random_val_matches[:], transform=None)
test_dataset_short = SARSegmentationDataset120(lmdb_path, random_test_matches[:], transform=None)

train_loader_short = DataLoader(train_dataset_short, batch_size=batch_size, shuffle=True)
val_loader_short = DataLoader(val_dataset_short, batch_size=batch_size, shuffle=False)
test_loader_short = DataLoader(test_dataset_short, batch_size=batch_size, shuffle=False)

Opening LMDB environment ...
Opening LMDB environment ...
Opening LMDB environment ...


## Tensorflow Board

In [34]:
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir=runs --port=6010

Reusing TensorBoard on port 6010 (pid 14208), started 14:03:58 ago. (Use '!kill 14208' to kill it.)

# Training

In [32]:
want_to_train120 = True

model = load_base_with_bigearth_pretrained120()
# model = load_from_checkpoint120("../models/unet120_full_epoch_2.pth")

# Freeze encoder layers
for param in model.encoder.parameters():
    param.requires_grad = False
    
if want_to_train120:
    criterion_base = FocalLoss(mode='multiclass', ignore_index=20)  
    # model = create_base_model()
    training(
        model = model, 
        epoch_start = 1,
        epoch_end =  20,
        loss_fn = criterion_base,
        train_loader = train_loader,
        val_loader = val_loader,
        num_classes = 20,
        lr = 1e-4,
        model_name = "unet120_full",)
    # for param in model.encoder.parameters():
    #     param.requires_grad = False

else:
    print("Not training")

c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\configilm\ConfigILM.py:134: UserWarning: Keyword 'img_size' unknown. Trying to ignore and restart creation.
  warnings.warn(f"Keyword '{failed_kw}' unknown. Trying to ignore and restart creation.")
c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training started for unet120_full from epoch 1 to 20


Epoch 1/20:   0%|          | 0/14867 [00:00<?, ?batch/s]c:\Users\Jean\anaconda3\envs\ActivationPrototypes_SARSeg\lib\site-packages\torch\nn\modules\module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
Epoch 1/20: 100%|██████████| 14867/14867 [22:49<00:00, 10.86batch/s, f1=0.48, iou=0.32, loss=3.61]  


Epoch 1, Train Loss: 3.6131, IoU: 0.3202, F1: 0.4796


Validation: 100%|██████████| 7647/7647 [08:15<00:00, 15.43batch/s, f1=0.514, iou=0.383, loss=3.59]


...

Epoch 19, Val Loss: 3.5426, IoU: 0.4728, F1: 0.6151


Epoch 20/20: 100%|██████████| 14867/14867 [25:24<00:00,  9.75batch/s, f1=0.653, iou=0.489, loss=3.52] 


Epoch 20, Train Loss: 3.5220, IoU: 0.4891, F1: 0.6530


Validation: 100%|██████████| 7647/7647 [08:12<00:00, 15.52batch/s, f1=0.596, iou=0.455, loss=3.55]


Epoch 20, Val Loss: 3.5531, IoU: 0.4554, F1: 0.5963
